In [1]:
import sys
import numpy as np
import pandas as pd

In [2]:
from pathlib import Path

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)


# Ajouter la racine du projet au chemin Python
ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

In [3]:
# Importation des fonctions de préparation des données
from donnees import (
    charger_compteurs,
    parties,
    seuil,
    ecrire_predictions,
    COLONNES_E
)

# Importation de notre classe Invariant Mining
from algorithmes.invariant_mining import InvariantMining, score

In [4]:
# Chargement
df = charger_compteurs()
p = parties(df)

display(df.head())

print("Dimensions :", df.shape)

print("\nRépartition des données :")
print(df["split"].value_counts())

print("\nTaux d'anomalies :")
print(df.groupby("split")["label"].mean())

,block_id,label,split,E1,E2,E3,E4,E5,E6,E7,...,E26,E27,E28,E29,E30,n_warn,n_error,length,n_unique,duration_s
0,blk_-1608999687919862906,0,train,0,0,203,0,10,7,0,...,10,0,0,0,0,0,0,269,12,3802
1,blk_7503483334202473044,0,train,0,2,1,0,3,0,0,...,3,0,0,0,0,0,0,22,9,3802
2,blk_-3544583377289625738,1,train,0,0,203,0,3,0,0,...,3,0,0,0,0,1,0,223,9,3797
3,blk_-9073992586687739851,0,train,0,3,0,0,3,0,0,...,3,0,0,0,0,0,0,22,8,50448
4,blk_7854771516489510256,0,train,0,3,1,15,3,0,0,...,3,0,0,0,0,15,0,38,10,50583


Dimensions : (575061, 38)

Répartition des données :
split
train    402542
test      86260
val       86259
Name: count, dtype: int64

Taux d'anomalies :
split
test     0.012938
train    0.032580
val      0.030223
Name: label, dtype: float64


In [5]:
COLONNES_FEATURES = (
        list(COLONNES_E)
        + [
            "n_warn",
            "n_error",
            "length",
            "n_unique",
            "duration_s"
        ]
)

# Vérification des colonnes
colonnes_manquantes = [
    c for c in COLONNES_FEATURES
    if c not in df.columns
]

if colonnes_manquantes:
    raise ValueError(
        f"Colonnes manquantes : {colonnes_manquantes}"
    )

print("Nombre de caractéristiques :", len(COLONNES_FEATURES))
print("Caractéristiques :", COLONNES_FEATURES)

Nombre de caractéristiques : 35
Caractéristiques : ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'E25', 'E26', 'E27', 'E28', 'E29', 'E30', 'n_warn', 'n_error', 'length', 'n_unique', 'duration_s']


In [6]:
# Entraînement : uniquement les blocs normaux
X_train = p["train_normal"][COLONNES_FEATURES].to_numpy(dtype=float)

# Validation : normales + anomalies
X_val = p["val"][COLONNES_FEATURES].to_numpy(dtype=float)
y_val = p["val"]["label"].to_numpy()

# Validation normale : calibration du seuil
X_val_normal = p["val_normal"][COLONNES_FEATURES].to_numpy(dtype=float)

# Test final
X_test = p["test"][COLONNES_FEATURES].to_numpy(dtype=float)
y_test = p["test"]["label"].to_numpy()

print("Train normal       :", X_train.shape)
print("Validation         :", X_val.shape)
print("Validation normale :", X_val_normal.shape)
print("Test               :", X_test.shape)

# Vérification des valeurs numériques
for nom, X in [
    ("Train", X_train),
    ("Validation", X_val),
    ("Validation normale", X_val_normal),
    ("Test", X_test)
]:
    assert np.isfinite(X).all(), f"Valeurs invalides dans {nom}"

print("\nToutes les données sont numériques et valides.")

Train normal       : (389427, 35)
Validation         : (86259, 35)
Validation normale : (83652, 35)
Test               : (86260, 35)

Toutes les données sont numériques et valides.


In [7]:
from itertools import product

valeurs_r2 = [0.90, 0.95, 0.98, 0.99, 0.995]
valeurs_invariants = [5, 10, 15, 20]
aggregations = ["mean", "max"]

reglages = [
    {
        "min_r2": r2,
        "max_invariants": n,
        "aggregation": agg
    }
    for r2, n, agg in product(
        valeurs_r2,
        valeurs_invariants,
        aggregations
    )
]

print("Nombre de configurations :", len(reglages))

Nombre de configurations : 40


In [8]:
resultats = []

meilleur_f1 = -1
meilleur_precision = -1
meilleur_reglage = None

for i, reglage in enumerate(reglages, 1):

    print(f"\nConfiguration {i}/{len(reglages)}")
    print("Réglages :", reglage)

    try:
        # 1. Création et entraînement
        model = InvariantMining(**reglage)
        model.fit(X_train)

        # 2. Calibration du seuil sur les normales
        scores_normal = score(model, X_val_normal)
        s = seuil(scores_normal)

        # 3. Prédictions sur toute la validation
        scores_val = score(model, X_val)
        y_pred = (scores_val > s).astype(int)

        # 4. Calcul des métriques
        precision = precision_score(
            y_val, y_pred, zero_division=0
        )
        recall = recall_score(
            y_val, y_pred, zero_division=0
        )
        f1 = f1_score(
            y_val, y_pred, zero_division=0
        )

        tn, fp, fn, tp = confusion_matrix(
            y_val, y_pred
        ).ravel()

        resultats.append({
            "min_r2": reglage["min_r2"],
            "max_invariants": reglage["max_invariants"],
            "aggregation": reglage["aggregation"],
            "nombre_invariants": len(model.invariants),
            "seuil": s,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "FP": fp,
            "FN": fn
        })

        print("Nombre invariants :", len(model.invariants))
        print("Seuil             :", round(s, 5))
        print("Precision         :", round(precision, 4))
        print("Recall            :", round(recall, 4))
        print("F1-score          :", round(f1, 4))
        print("Faux positifs     :", fp)
        print("Faux négatifs     :", fn)

        # 5. Sélection : F1 prioritaire, précision en cas d'égalité
        if (
            f1 > meilleur_f1
            or (
                np.isclose(f1, meilleur_f1)
                and precision > meilleur_precision
            )
        ):
            meilleur_f1 = f1
            meilleur_precision = precision
            meilleur_reglage = reglage.copy()

    except ValueError as e:
        print("Configuration ignorée :", e)


if meilleur_reglage is None:
    raise RuntimeError("Aucune configuration valide.")

# Tableau comparatif
df_resultats = pd.DataFrame(resultats)

df_resultats = df_resultats.sort_values(
    by=["f1", "precision", "recall"],
    ascending=False
).reset_index(drop=True)

display(df_resultats)

print("\nMEILLEURE CONFIGURATION")
print("Paramètres :", meilleur_reglage)
print("F1 validation :", round(meilleur_f1, 4))



Configuration 1/40
Réglages : {'min_r2': 0.9, 'max_invariants': 5, 'aggregation': 'mean'}
Nombre d'invariants découverts : 5
R² moyen : 1.0
Nombre invariants : 5
Seuil             : 1.938
Precision         : 0.8615
Recall            : 1.0
F1-score          : 0.9256
Faux positifs     : 419
Faux négatifs     : 0

Configuration 2/40
Réglages : {'min_r2': 0.9, 'max_invariants': 5, 'aggregation': 'max'}
Nombre d'invariants découverts : 5
R² moyen : 1.0
Nombre invariants : 5
Seuil             : 2.20439
Precision         : 0.8615
Recall            : 1.0
F1-score          : 0.9256
Faux positifs     : 419
Faux négatifs     : 0

Configuration 3/40
Réglages : {'min_r2': 0.9, 'max_invariants': 10, 'aggregation': 'mean'}
Nombre d'invariants découverts : 10
R² moyen : 0.9996
Nombre invariants : 10
Seuil             : 1.57965
Precision         : 0.8615
Recall            : 1.0
F1-score          : 0.9256
Faux positifs     : 419
Faux négatifs     : 0

Configuration 4/40
Réglages : {'min_r2': 0.9, 'max_

,min_r2,max_invariants,aggregation,nombre_invariants,seuil,precision,recall,f1,FP,FN
0,0.900,5,mean,5,1.938004,0.861533,1.0,0.925617,419,0
1,0.900,5,max,5,2.204389,0.861533,1.0,0.925617,419,0
2,0.900,10,mean,10,1.579652,0.861533,1.0,0.925617,419,0
3,0.900,10,max,10,2.224745,0.861533,1.0,0.925617,419,0
4,0.900,15,mean,13,1.715637,0.861533,1.0,0.925617,419,0
5,0.900,15,max,13,2.970636,0.861533,1.0,0.925617,419,0
6,0.900,20,mean,13,1.715637,0.861533,1.0,0.925617,419,0
7,0.900,20,max,13,2.970636,0.861533,1.0,0.925617,419,0
8,0.950,5,mean,5,1.938004,0.861533,1.0,0.925617,419,0
9,0.950,5,max,5,2.204389,0.861533,1.0,0.925617,419,0



MEILLEURE CONFIGURATION
Paramètres : {'min_r2': 0.9, 'max_invariants': 5, 'aggregation': 'mean'}
F1 validation : 0.9256


In [9]:
# Création du modèle final
model_final = InvariantMining(**meilleur_reglage)

# Entraînement sur les données normales de train
model_final.fit(X_train)

# Calibration du seuil sur les données normales de validation
scores_val_normal = score(model_final, X_val_normal)
seuil_final = seuil(scores_val_normal)

print("Seuil final :", seuil_final)

# Calcul des scores sur le test
scores_test = score(model_final, X_test)

# Classification
y_pred_test = (scores_test > seuil_final).astype(int)

# Évaluation finale
print("\n RÉSULTATS TEST ")

print("Accuracy  :", round(
    accuracy_score(y_test, y_pred_test), 4
))

print("Precision :", round(
    precision_score(y_test, y_pred_test, zero_division=0), 4
))

print("Recall    :", round(
    recall_score(y_test, y_pred_test, zero_division=0), 4
))

print("F1-score  :", round(
    f1_score(y_test, y_pred_test, zero_division=0), 4
))

print("ROC-AUC   :", round(
    roc_auc_score(y_test, scores_test), 4
))

# Export des prédictions
Path("../../results").mkdir(
    parents=True,
    exist_ok=True
)

ecrire_predictions(
    p["test"],
    scores_test,
    seuil_final,
    "../../results/invariant_mining_test.csv"
)

print("\nPrédictions enregistrées dans :")
print("../../results/invariant_mining_test.csv")

Nombre d'invariants découverts : 5
R² moyen : 1.0
Seuil final : 1.9380040438311725

 RÉSULTATS TEST 
Accuracy  : 0.9948
Precision : 0.714
Recall    : 1.0
F1-score  : 0.8331
ROC-AUC   : 0.9999

Prédictions enregistrées dans :
../../results/invariant_mining_test.csv


In [10]:
fichier = Path("../../results/invariant_mining_test.csv")

df_predictions = pd.read_csv(fichier)

print("Colonnes :", df_predictions.columns.tolist())
print("Nombre de lignes :", len(df_predictions))
print("Valeurs de pred :", df_predictions["pred"].value_counts())

display(df_predictions.head())

Colonnes : ['block_id', 'score', 'pred']
Nombre de lignes : 86260
Valeurs de pred : pred
0    84697
1     1563
Name: count, dtype: int64


,block_id,score,pred
0,blk_3458545012440520721,0.054919,0
1,blk_-1160437354072569825,0.055222,0
2,blk_-8961379332018832424,0.055197,0
3,blk_-7458466477853961957,0.055222,0
4,blk_3070076663445833576,0.055197,0


In [11]:
import time

debut_train = time.perf_counter()

model_final = InvariantMining(**meilleur_reglage)
model_final.fit(X_train)

train_time = (time.perf_counter() - debut_train) / 60

print(f"Temps d'entraînement : {train_time:.2f} minutes")

Nombre d'invariants découverts : 5
R² moyen : 1.0
Temps d'entraînement : 0.18 minutes


In [12]:
debut_pred = time.perf_counter()

scores_test = score(model_final, X_test)
y_pred_test = (scores_test > seuil_final).astype(int)

pred_time = time.perf_counter() - debut_pred

print(f"Temps de prédiction : {pred_time:.2f} secondes")

Temps de prédiction : 0.16 secondes


In [13]:
import joblib

# Chemin du dossier contenant le notebook
DOSSIER_NOTEBOOK = Path.cwd()

# Fichier de sauvegarde
chemin_modele = DOSSIER_NOTEBOOK / "invariant_mining_model.pkl"

# Sauvegarder le modèle, le seuil et les paramètres
joblib.dump({
    "model": model_final,
    "threshold": seuil_final,
    "features": COLONNES_FEATURES,
    "best_params": meilleur_reglage
}, chemin_modele)

print("Modèle sauvegardé avec succès !")
print("Emplacement :", chemin_modele)

Modèle sauvegardé avec succès !
Emplacement : C:\Users\Oumnia\Documents\LOGsentinel\hdfs_v1\members\oumnia\invariant_mining_model.pkl
